# Vietnamese Social Media Text Processing, Embedding & Clustering Pipeline

This Google Colab / Jupyter notebook combines the end-to-end NLP & clustering pipeline from the project scripts (excluding `visualization_pca` and `visualization_tsne`):

1. **Concatenate Raw CSVs** (`concat_csv.py`) — Combine daily scraped CSV files into a single dataset.
2. **Vietnamese Text Preprocessing** (`preprocess/preprocess.py` + `preprocess/teencode.txt`) — Filter out non-Vietnamese posts, strip emojis/icons, and normalize Vietnamese teencode/slang while preserving URLs, mentions, hashtags, and capitalization.
3. **Word Segmentation & Sentence Embeddings** (`embedding/generate_embeddings.py`) — Segment Vietnamese compound words with `pyvi.ViTokenizer` and generate 768D dense vectors using `dangvantuan/vietnamese-embedding`.
4. **Evaluation Metrics & Topic Extraction** (`clustering/evaluation/metrics.py`, `clustering/evaluation/topic_extractor.py`) — Compute intrinsic clustering quality metrics (Silhouette, Davies-Bouldin, Calinski-Harabasz, Intra-Cluster Cosine) and extract distinctive keywords via Class-based TF-IDF (`c-TF-IDF`).
5. **Clustering Benchmark Suite & Interactive Visualization** (`clustering/run_benchmark.py`) — Benchmark 5 clustering families (Spherical K-Means, Agglomerative Hierarchical, HDBSCAN Direct & UMAP, GMM with PCA-20D, BERTopic), rank models, and visualize the winning topics.

## 1. Environment Setup & Configuration

Run the cell below to install required dependencies (especially in Google Colab) and configure project paths.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

# Install missing dependencies automatically in Colab (or if missing locally)
REQUIRED_PACKAGES = {
    "pyvi": "pyvi",
    "sentence_transformers": "sentence-transformers",
    "torch": "torch",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "hdbscan": "hdbscan",
    "umap": "umap-learn",
    "bertopic": "bertopic",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
}

missing = []
for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing missing packages: {missing}")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
# Optional: Mount Google Drive when running in Google Colab
USE_GOOGLE_DRIVE = False  # Set to True if your data/project folder is on Google Drive

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/MLProject"  # Adjust to your Drive folder
elif IN_COLAB:
    PROJECT_DIR = "/content/MLProject"
else:
    PROJECT_DIR = os.getcwd()

os.makedirs(PROJECT_DIR, exist_ok=True)

# Pipeline paths & settings
DATA_DIR = os.path.join(PROJECT_DIR, "data")
RAW_COMBINED_CSV = os.path.join(DATA_DIR, "combined.csv")
PREPROCESSED_CSV = os.path.join(DATA_DIR, "combined_preprocessed.csv")
TEENCODE_FILE = os.path.join(PROJECT_DIR, "preprocess", "teencode.txt")
EMBEDDING_DIR = os.path.join(PROJECT_DIR, "embedding")
CLUSTERING_OUTPUT_DIR = os.path.join(PROJECT_DIR, "clustering", "output")

TEXT_COLUMN = "text"
MODEL_NAME = "dangvantuan/vietnamese-embedding"
BATCH_SIZE = 32
DEVICE = None          # None = auto-detect ("cuda", "mps", or "cpu")
NORMALIZE_EMBEDDINGS = True
FILTER_NON_VN = True
RANDOM_STATE = 42
OFFLINE_MODE = False   # Keep False in Colab to allow downloading HuggingFace model

if OFFLINE_MODE and not os.environ.get("HF_TOKEN"):
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.dirname(TEENCODE_FILE), exist_ok=True)
os.makedirs(EMBEDDING_DIR, exist_ok=True)
os.makedirs(CLUSTERING_OUTPUT_DIR, exist_ok=True)

print(f"PROJECT_DIR:           {PROJECT_DIR}")
print(f"DATA_DIR:              {DATA_DIR}")
print(f"TEENCODE_FILE:         {TEENCODE_FILE}")
print(f"EMBEDDING_DIR:         {EMBEDDING_DIR}")
print(f"CLUSTERING_OUTPUT_DIR: {CLUSTERING_OUTPUT_DIR}")

In [ ]:
# Optional: Upload CSV files directly in Google Colab (if not using Drive or git clone)
UPLOAD_CSV_IN_COLAB = False  # Set to True to trigger Colab file upload into DATA_DIR

if IN_COLAB and UPLOAD_CSV_IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for fname, content in uploaded.items():
        dest = os.path.join(DATA_DIR, fname)
        with open(dest, "wb") as f:
            f.write(content)
        print(f"Saved uploaded file to: {dest}")

## 2. Step 1 — Concatenate Raw CSV Files (`concat_csv.py`)

Combines all raw `.csv` files in `DATA_DIR` into `data/combined.csv` using Python's built-in `csv` module.

In [ ]:
import csv
import glob


def concatenate_csv_files(folder_path="./", output_file="combined.csv", exclude_files=None):
    """
    Concatenate all CSV files in the specified folder into a single output file.
    Uses Python's built-in csv module (no external dependencies required).
    """
    folder = os.path.abspath(folder_path)
    print(folder)
    output_path = os.path.abspath(os.path.join(folder, output_file))
    excluded_paths = {output_path}
    if exclude_files:
        for fname in exclude_files:
            excluded_paths.add(os.path.abspath(os.path.join(folder, fname)))

    # Find all CSV files in folder except excluded files
    pattern = os.path.join(folder, "*.csv")
    csv_files = sorted(
        [f for f in glob.glob(pattern) if os.path.abspath(f) not in excluded_paths]
    )

    if not csv_files:
        print(f"No raw CSV files found in '{folder}'.")
        return None

    print(f"Found {len(csv_files)} CSV file(s) to combine:")
    for f in csv_files:
        print(f"  - {os.path.basename(f)}")

    total_rows = 0
    header_written = False

    with open(output_path, "w", newline="", encoding="utf-8-sig") as out_f:
        writer = csv.writer(out_f)

        for file_path in csv_files:
            file_name = os.path.basename(file_path)
            with open(file_path, "r", newline="", encoding="utf-8", errors="replace") as in_f:
                reader = csv.reader(in_f)
                try:
                    header = next(reader)
                except StopIteration:
                    # Empty file
                    print(f"Skipping empty file: {file_name}")
                    continue

                if not header_written:
                    writer.writerow(header)
                    header_written = True

                file_row_count = 0
                for row in reader:
                    writer.writerow(row)
                    file_row_count += 1

                print(f"  Processed {file_name}: {file_row_count} rows")
                total_rows += file_row_count

    print(f"\nDone! Successfully combined {len(csv_files)} files ({total_rows} data rows).")
    print(f"Output saved to: {output_path}")
    return output_path

In [ ]:
# Run CSV concatenation (skips cleanly if only combined.csv is present)
raw_candidates = [
    f for f in glob.glob(os.path.join(DATA_DIR, "*.csv"))
    if os.path.basename(f) not in {os.path.basename(RAW_COMBINED_CSV), os.path.basename(PREPROCESSED_CSV)}
]

if raw_candidates:
    concatenate_csv_files(
        folder_path=DATA_DIR,
        output_file=os.path.basename(RAW_COMBINED_CSV),
        exclude_files=[os.path.basename(PREPROCESSED_CSV)],
    )
elif os.path.exists(RAW_COMBINED_CSV):
    print(f"Using existing combined CSV: {RAW_COMBINED_CSV}")
else:
    print(f"No CSV files found in {DATA_DIR}. Please upload CSV files to DATA_DIR first.")

## 3. Step 2 — Vietnamese Text Preprocessing (`preprocess/preprocess.py`)

Preprocesses the combined CSV by:
1. Detecting and excluding non-Vietnamese posts (foreign languages, pure emojis/punctuation, empty posts).
2. Removing all icons, emojis, emoticons, pictographs, and symbol artifacts.
3. Converting Vietnamese teencode / slang into standard Vietnamese words using `teencode.txt`.
4. Preserving URLs, domains, emails, mentions, hashtags, and token capitalization.

*(Includes the built-in `teencode.txt` dictionary as an automatic fallback if running standalone in Colab.)*

In [ ]:
import re
import time
from typing import Dict, List, Optional, Tuple

# Comprehensive Unicode Emoji / Icon Pattern
ICON_PATTERN = re.compile(
    r'('
    r'[\U0001F600-\U0001F64F]'  # Emoticons
    r'|[\U0001F300-\U0001F5FF]'  # Misc Symbols and Pictographs
    r'|[\U0001F680-\U0001F6FF]'  # Transport and Map
    r'|[\U0001F700-\U0001F77F]'  # Alchemical Symbols
    r'|[\U0001F780-\U0001F7FF]'  # Geometric Shapes Extended
    r'|[\U0001F800-\U0001F8FF]'  # Supplemental Arrows-C
    r'|[\U0001F900-\U0001F9FF]'  # Supplemental Symbols and Pictographs
    r'|[\U0001FA00-\U0001FA6F]'  # Chess Symbols / Extended-A
    r'|[\U0001FA70-\U0001FAFF]'  # Symbols and Pictographs Extended-A
    r'|[\U0001FB00-\U0001FBFF]'  # Symbols Extended-B
    r'|[\U0001F1E6-\U0001F1FF]'  # Regional indicator symbols (country flags)
    r'|[\U0001F3FB-\U0001F3FF]'  # Emoji modifier skin tones
    r'|[\u2600-\u26FF]'          # Misc symbols
    r'|[\u2700-\u27BF]'          # Dingbats
    r'|[\u2300-\u23FF]'          # Misc Technical
    r'|[\u2B00-\u2BFF]'          # Misc Symbols and Arrows
    r'|[\u2580-\u259F]'          # Block elements
    r'|[\u25A0-\u25FF]'          # Geometric shapes
    r'|[\u2190-\u21FF]'          # Arrows
    r'|[\u203C\u2049\u2122\u2139\u00A9\u00AE]'  # Special symbols
    r'|[\u200D\uFE0E\uFE0F]'     # ZWJ and variation selectors
    r'|[0-9#*]\uFE0F?\u20E3'     # Enclosing Keycaps
    r')+',
    flags=re.UNICODE
)

# Pattern to protect structured entities (URLs, domains, emails, handles, hashtags)
PROTECT_PATTERN = (
    r'https?://\S+'
    r'|www\.\S+'
    r'|[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?:/\S*)?'
    r'|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
    r'|@[\w.-]+'
    r'|#[\w.-]+'
)

# Vietnamese diacritics matcher
VN_DIACRITICS_RE = re.compile(
    r'[àáảãạâầấẩẫậăằắẳẵặèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ'
    r'ÀÁẢÃẠÂẦẤẨẪẬĂẰẮẲẴẶÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴĐ]'
)

# Foreign non-Latin scripts (CJK, Korean, Japanese, Cyrillic, Arabic, Thai)
NON_LATIN_RE = re.compile(
    r'[\u4e00-\u9fff\u3400-\u4dbf\uac00-\ud7af\u3040-\u30ff\u0400-\u04ff\u0600-\u06ff\u0e00-\u0e7f]'
)

# Common high-frequency English stopwords for language disambiguation
COMMON_EN_WORDS = {
    'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'i', 'it', 'for', 'not', 'on', 'with',
    'he', 'as', 'you', 'do', 'at', 'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her',
    'she', 'or', 'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their', 'what', 'so', 'up',
    'out', 'if', 'about', 'who', 'get', 'which', 'go', 'me', 'when', 'make', 'can', 'like', 'time',
    'no', 'just', 'him', 'know', 'take', 'people', 'into', 'year', 'your', 'good', 'some', 'could',
    'them', 'see', 'other', 'than', 'then', 'now', 'look', 'only', 'come', 'its', 'over', 'think',
    'also', 'back', 'after', 'use', 'two', 'how', 'our', 'work', 'first', 'well', 'way', 'even',
    'new', 'want', 'because', 'any', 'these', 'give', 'day', 'most', 'us', 'is', 'are', 'was', 'were'
}

# Common Vietnamese unaccented words and teencode/slang for accentless text detection
COMMON_VN_UNACCENTED = {
    'ko', 'k', 'khong', 'dc', 'đc', 'nguoi', 'ng', 'minh', 'toi', 'tui', 'tuii', 'ban', 'nay',
    'do', 'o', 'la', 'va', 'co', 'muon', 'thoi', 'nhe', 'nha', 'roi', 'chua', 'biet', 'thay',
    'nhieu', 'it', 'dep', 'xau', 'dung', 'sai', 'lam', 'gi', 'sao', 'nao', 'the', 'di', 've',
    'em', 'anh', 'chi', 'bac', 'ong', 'ba', 'tao', 'to', 'may', 'no', 'ho', 'chung', 'ai',
    'troi', 'oi', 'dcm', 'vcl', 'bcuoi', 'vl', 'clb', 'nckh', 'khum', 'hong', 'hongg', 'nx',
    'chx', 'trg', 'hc', 'cs', 'kbt', 'dhs', 'khs', 'v~', 'z', 'zậy', 'zui', 'ze', 'hok'
}

DEFAULT_TEENCODE_TSV = 'ctrai\tcon trai\nkhôg\tkhông\nbme\tbố mẹ\ncta\tchúng ta\nmih\tmình\nmqh\tmối quan hệ\ncgai\tcon gái\nnhữg\tnhững\nmng\tmọi người\nsvtn\tsinh viên tình nguyện\nr\trồi\nqtam\tquan tâm\nthươg\tthương\nqtâm\tquan tâm\nchug\tchung\ntrườg\ttrường\nthoy\tthôi\nđki\tđăng ký\natsm\tảo tưởng sức mạnh\nạk\tạ\ncv\tcông việc\nvch\tvãi chưởng\ncùg\tcùng\npn\tbạn\npjt\tbiết\nthjk\tthích\nktra\tkiểm tra\nnek\tnè\ncgái\tcon gái\nnthe\tnhư thế\nchúg\tchúng\nkái\tcái\ntìh\ttình\nphòg\tphòng\nlòg\tlòng\ntừg\ttừng\nrằg\trằng\nsốg\tsống\nthuj\tthôi\nthuơng\tthương\ncàg\tcàng\nđky\tđăng ký\nbằg\tbằng\nsviên\tsinh viên\nák\tá\nđág\tđáng\nnvay\tnhư vậy\nnhjeu\tnhiều\nxg\txuống\nzồi\trồi\ntrag\ttrang\nzữ\tdữ\natrai\tanh trai\nkte\tkinh tế\nđộg\tđộng\nlmht\tliên minh huyền thoại\ngắg\tgắng\nđzai\tđẹp trai\nthgian\tthời gian\nđồg\tđồng\nbtrai\tbạn trai\nnthê\tnhư thế\nhìhì\thì hì\nvọg\tvọng\nhihe\thi he\nđôg\tđông\nrăg\trăng\nthườg\tthường\ntcảm\ttình cảm\nđứg\tđứng\nksao\tkhông sao\ndz\tđẹp trai\nhjxhjx\thix hix\ncmày\tchúng mày\nxuốg\txuống\nnkư\tnhư\nlquan\tliên quan\ntiếg\ttiếng\nxih\txinh\nhìh\thình\nthàh\tthành\nngke\tnghe\ndzậy\tdậy\ntnào\tthế nào\ntưởg\ttưởng\nctrinh\tchương trình\nphog\tphong\nhôg\tkhông\nzìa\tgì\nkũg\tcũng\nntnao\tnhư thế nào\ntrọg\ttrọng\nnthế\tnhư thế\nnăg\tnăng\nngđó\tngười đó\nlquen\tlàm quen\nriêg\triêng\nngag\tngang\nhêhê\thê hê\nbnhiu\tbao nhiêu\nngốk\tngốc\nkậu\tcậu\nkqua\tkết quả\nhtrc\thôm trước\nđịh\tđịnh\ngđình\tgia đinh\ngiốg\tgiống\ncsống\tcuộc sống\nxug\txùng\nzùi\trồi\nbnhiêu\tbao nhiêu\ncbị\tchuẩn bị\nkòn\tcòn\nbuôg\tbuông\ncsong\tcuộc sống\nchàg\tchàng\nchăg\tchăng\nngàh\tngành\nllac\tliên lạc\nnkưng\tnhưng\nnắg\tnắng\ntíh\ttính\nkhoảg\tkhoảng\nthík\tthích\nngđo\tngười đó\nngkhác\tngười khác\nthẳg\tthẳng\nkảm\tcảm\ndàh\tdành\njúp\tgiúp\nlặg\tlặng\nvđê\tvấn đề\nbbè\tbạn bè\nbóg\tbóng\ndky\tđăng ký\ndòg\tdòng\nuốg\tuống\ntyêu\ttình yêu\nsnvv\tsinh nhật vui vẻ\nđthoại\tđiện thoại\nqhe\tquan hệ\ncviec\tcông việc\ntượg\ttượng\nqà\tquà\nthjc\tthích\nnhưq\tnhưng\ncđời\tcuộc đời\nbthường\tbình thường\nzà\tgià\nđáh\tđánh\nxloi\txin lỗi\nzám\tdám\nqtrọng\tquan trọng\nbìh\tbình\nlzi\tlàm gì\nqhệ\tquan hệ\nđhbkhn\tđại học bách khoa hà nội\nkủa\tcủa\nlz\tlàm gì\nđhkhtn\tđại học khoa học tự nhiên\nđóg\tđóng\ncka\tcha\nlgi\tlàm gì\nnvậy\tnhư vậy\nqả\tquả\nđkiện\tđiều kiện\nnèk\tnè\ntlai\ttương lai\nbsĩ\tbác sĩ\nhkì\thọc kỳ\nđcsvn\tđảng cộng sản việt nam\nvde\tvấn đề\nchta\tchúng ta\nòy\trồi\nltinh\tlinh tinh\nngyeu\tngười yêu\nđthoai\tđiện thoại\nsnghĩ\tsuy nghĩ\nnặg\tnặng\nhọk\thọc\ndừg\tdừng\nhphúc\thạnh phúc\nhiha\thi ha\nwtâm\tquan tâm\nthíck\tthích\nchuện\tchuyện\nlạh\tlạnh\nfây\tphây\nntnày\tnhư thế này\nlúk\tlúc\nhaj\thai\nngía\tnghía\nmớj\tmới\nhsơ\thồ sơ\nctraj\tcon trai\nnyêu\tngười yêu\nđiiiiiii\tđi\nrồii\trồi\nkih\tkinh\nkb\tkết bạn\nhixxx\thích\ndthương\tdễ thương\nnhiềuuu\tnhiều\nctrình\tchương trình\nmìnk\tmình\nmjh\tmình\nng\tngười\nvc\tvợ chồng\nuhm\từm\nthỳ\tthì\nnyc\tngười yêu cũ\ntks\tthanks\nnàg\tnàng\nthôii\tthôi\nđjên\tđiên\nbgái\tbạn gái\nvớii\tvới\nxink\txinh\nhđộng\thành động\nđhọc\tđại học\nmk\tmình\nbn\tbạn\nthik\tthích\ncj\tchị\nmn\tmọi người\nnguoi\tngười\nnógn\tnóng\nhok\tkhông\nko\tkhông\nbik\tbiết\nvs\tvới\ncx\tcũng\nmik\tmình\nwtf\twhat the fuck\nđc\tđược\ncmt\tcomment\nck\tchồng\nchk\tchồng\nngta\tngười ta\ngđ\tgia đình\noh\tồ\nvk\tvợ\nctác\tcông tác\nsg\tsài gòn\nae\tanh em\nah\tà\nạh\tạ\nrì\tgì\nms\tmới\nvn\tviệt nam\nnhaa\tnha\ncũg\tcũng\nđag\tđang\nơiii\tơi\nace\tanh chị em\nàk\tà\nuh\từ\ncmm\tcon mẹ mày\ncmnr\tcon mẹ nó rồi\nơiiii\tơi\nhnay\thôm nay\nukm\từm\ntq\ttrung quốc\nctr\tchương trình\nđii\tđi\nnch\tnói chuyện\ntrieu\ttriệu\nhahah\tha ha\nnta\tngười ta\nngèo\tnghèo\nkêh\tkênh\nak\tà\nad\tadmin\nj\tgì\nny\tngười yêu\ndc\tđược\nqc\tquảng cáo\nbaoh\tbao giờ\nzui\tvui\nzẻ\tvẻ\ntym\ttim\naye\tanh yêu em\neya\tem yêu anh\nfb\tfacebook\ninsta\tinstagram\nz\tvậy\nthich\tthích\nvcl\tvờ cờ lờ\nđt\tđiện thoại\nacc\taccount\nlol\tlồn\nloz\tlồn\nlozz\tlồn\ntrc\ttrước\nchs\tchẳng hiểu sao\nđhs\tđéo hiểu sao\nqá\tquá\nntn\tnhư thế nào\nwá\tquá\nzậy\tvậy\nzô\tvô\nytb\tyoutube\nvđ\tvãi đái\nvchg\tvãi chưởng\nsml\tsấp mặt lờ\nxl\txin lỗi\ncmn\tcon mẹ nó\nhjhj\thi hi\nvv\tvui vẻ\nns\tnói\niu\tyêu\nvcđ\tvãi cải đái\nin4\tinfo\nqq\tquằn què\nkh\tkhông\nzạ\tvậy\noy\trồi\njo\tgiờ\nclmm\tcái lồn mẹ mày\nbsvv\tbuổi sáng vui vẻ\ntroai\ttrai\nwa\tquá\nhjx\thix\ne\tem\nik\tý\nji\tgì\nce\tchị em\nlm\tlàm\nđz\tđẹp giai\nsr\tsorry\nib\tinbox\nhoy\tthôi\nđbh\tđéo bao giờ\nk\tkhông\nvd\tví dụ\ncũng z\tcũng vậy\nz là\tvậy là\nunf\tunfriend\nmy fen\tmy friend\nfen\tfriend\ncty\tcông ty\non lai\tonline\nu hai ba\tu23\nkô\tkhông\nđtqg\tđội tuyển quốc gia\nhqua\thôm qua\nxog\txong\nuh\từ\nuk\từ\nnhoé\tnhé\nbiet\tbiết\nquí\tquý\nstk\tsố tài khoản\nhong kong\thồng kông\nđươc\tđược\nnghành\tngành\nnvqs\tnghĩa vụ quân sự\nngừoi\tngười\ntrog\ttrong\ntgian\tthời gian\nbiêt\tbiết\nfải\tphải\nnguời\tngười\ntđn\tthế đéo nào\nbth\tbình thường\nvcđ\tvãi cả đái\ntgdd\tthế giới di động\nkhg\tkhông\nnhưg\tnhưng\nthpt\ttrung học phổ thông\nthằg\tthằng\nđuợc\tđược\ndc\tđược\nđc\tđược\nah\tà\nàh\tà\nku\tcu\nthým\tthím\nonl\tonline\nzô\tdô\nzú\tvú\ncmnd\tchứng minh nhân dân\nsđt\tsố điện thoại\nklq\tkhông liên quan\ntui\ttôi\ntuii\ttôi\nkhum\tkhông\nhong\tkhông\nhông\tkhông\nhổng\tkhông\nhongg\tkhông\nkbt\tkhông biết\ndhs\tđéo hiểu sao\nkhs\tkhông hiểu sao\nnx\tnữa\nchx\tchưa\ntrg\ttrường\nhc\thọc\nbcuoi\tbuồn cười\ntr oi\ttrời ơi\ntroi\ttrời\nrùi\trồi\nrui\trồi\nđme\tđịt mẹ\ndcm\tđịt con mẹ\nvkl\tvãi cả lồn\nvl\tvãi lồn\nclb\tcâu lạc bộ\nnckh\tnghiên cứu khoa học\ncap\tcaption\ntb\ttrung bình\np/s\ttái bút\nps\ttái bút\nb\tbạn\nt\ttôi\nm\tmày\nnt\tnhắn tin\nct\tchia tay\nđr\tđúng rồi\nsv\tsinh viên\nhs\thọc sinh\nhsg\thọc sinh giỏi\nthptqg\ttrung học phổ thông quốc gia\nktx\tký túc xá\nmtb\tmáy tính bảng\nrcm\trecommend\ncf\tcà phê\nnhg\tnhưng\nđh\tđại học\ngdtt\tgiao dịch trực tiếp\nnv\tnhân viên\ncg\tcũng\nbt\tbiết\ntk\ttài khoản\ntnhien\ttự nhiên\nđcm\tđịt con mẹ\nlun\tluôn\nzai\ttrai\nvaiz\tvãi\nhum\thôm\nhỉu\thiểu\nkhộ\tkhổ\nrấc\trất\nbít\tbiết\ndzui\tvui\nch\tchưa\nmụt\tmột\nbùn\tbuồn\nthặc\tthật\nchời\ttrời\ntrùi\ttrời\nkkk\tha ha ha\ntr ơi\ttrời ơi\ntr đấc cơi\ttrời đất ơi\nnc ngoài\tnước ngoài\nnc mắt\tnước mắt\n'

In [ ]:
def ensure_teencode_file(dict_path: str) -> str:
    """Ensure teencode.txt exists at dict_path; write built-in fallback if missing."""
    if not os.path.exists(dict_path):
        os.makedirs(os.path.dirname(os.path.abspath(dict_path)), exist_ok=True)
        with open(dict_path, "w", encoding="utf-8") as f:
            f.write(DEFAULT_TEENCODE_TSV)
        print(f"Created default teencode dictionary at: {dict_path}")
    return dict_path


def is_vietnamese(text: Optional[str]) -> bool:
    """
    Determine whether text is Vietnamese using a general heuristic:
    1. Rejects empty, whitespace-only, or pure link/mention texts.
    2. Recognizes texts containing Vietnamese diacritics (unless foreign non-Latin scripts dominate).
    3. For unaccented texts, checks vocabulary overlap between English and Vietnamese slang/stopwords.
    """
    if not text:
        return False

    # Remove URLs, mentions, hashtags
    clean = re.sub(r'https?://\S+|www\.\S+|@\S+|#\S+', ' ', text)

    # Extract words
    words = re.findall(
        r'[a-zA-Zàáảãạâầấẩẫậăằắẳẵặèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ'
        r'ÀÁẢÃẠÂẦẤẨẪẬĂẰẮẲẴẶÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴĐ]+',
        clean
    )
    if not words:
        return False

    has_vn_diacritics = bool(VN_DIACRITICS_RE.search(clean))
    non_latin_count = len(NON_LATIN_RE.findall(clean))

    # 1. Text with Vietnamese diacritics
    if has_vn_diacritics:
        vn_diacritic_count = len(VN_DIACRITICS_RE.findall(clean))
        if non_latin_count > vn_diacritic_count * 2:
            return False
        return True

    # 2. Text without Vietnamese diacritics
    if non_latin_count > 0:
        return False

    words_lower = [w.lower() for w in words]
    en_matches = sum(1 for w in words_lower if w in COMMON_EN_WORDS)
    vn_matches = sum(1 for w in words_lower if w in COMMON_VN_UNACCENTED)

    if vn_matches > en_matches:
        return True
    if en_matches > 0:
        return False

    return vn_matches > 0


def remove_icons(text: Optional[str]) -> str:
    """
    Remove all icons, emojis, pictographs, and related variation selectors from text.
    Also normalizes horizontal whitespace on each line, preserving line breaks.
    """
    if not text:
        return "" if text is not None else None

    cleaned = ICON_PATTERN.sub("", text)
    cleaned = re.sub(r"[\u200B-\u200D\uFEFF]", "", cleaned)
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in cleaned.split("\n")]
    return "\n".join(lines)


def load_teencode_dict(dict_path: str) -> Dict[str, str]:
    """
    Load teencode dictionary from a tab-separated file.
    Format: <teencode>\t<standard_word>
    """
    teencode_dict = {}
    if not os.path.exists(dict_path):
        raise FileNotFoundError(f"Teencode dictionary not found at: {dict_path}")

    with open(dict_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                key = parts[0].strip().lower()
                value = parts[1].strip()
                teencode_dict[key] = value

    return teencode_dict


def build_teencode_pattern(teencode_dict: Dict[str, str]) -> re.Pattern:
    """
    Build an optimized regular expression pattern matching teencode entries.
    Longer multi-word phrases are prioritized first to avoid partial replacements.
    Protected entities (URLs, emails, handles) are matched first to prevent mangling.
    """
    sorted_keys = sorted(
        teencode_dict.keys(),
        key=lambda x: (len(x.split()), len(x)),
        reverse=True
    )

    teencode_patterns = []
    for k in sorted_keys:
        words = [re.escape(w) for w in k.split()]
        teencode_patterns.append(r"\s+".join(words))

    full_pattern_str = (
        rf"(?P<protect>{PROTECT_PATTERN})"
        rf"|(?P<teencode>(?<![-\w])(?:{'|'.join(teencode_patterns)})(?![-\w]))"
    )

    return re.compile(full_pattern_str, flags=re.IGNORECASE)


def match_case(orig: str, replacement: str) -> str:
    """
    Match the capitalization style of the original token:
    - UPPERCASE: returns replacement in UPPERCASE
    - Capitalized / Titlecase: returns replacement with first letter capitalized
    - lowercase / other: returns replacement as is
    """
    if orig.isupper():
        return replacement.upper()
    elif orig and orig[0].isupper():
        return replacement[0].upper() + replacement[1:]
    return replacement


def convert_teencode(
    text: Optional[str],
    teencode_dict: Dict[str, str],
    pattern: re.Pattern
) -> str:
    """
    Convert Vietnamese teencode words to standard words using the compiled pattern
    and teencode dictionary while preserving protected entities and casing.
    """
    if not text:
        return "" if text is not None else None

    def _replace_match(match: re.Match) -> str:
        if match.group("protect"):
            return match.group("protect")

        matched_str = match.group("teencode")
        if not matched_str:
            return match.group(0)

        normalized_key = re.sub(r"\s+", " ", matched_str.lower().strip())
        standard_word = teencode_dict.get(normalized_key, matched_str)
        return match_case(matched_str, standard_word)

    return pattern.sub(_replace_match, text)


def preprocess_text(
    text: Optional[str],
    teencode_dict: Dict[str, str],
    pattern: re.Pattern
) -> str:
    """
    Execute full text preprocessing pipeline:
    1. Remove any icon/emoji.
    2. Convert Vietnamese teencode to standard Vietnamese words.
    """
    if text is None:
        return ""

    no_icons = remove_icons(text)
    return convert_teencode(no_icons, teencode_dict, pattern)


def preprocess_csv(
    input_file: str,
    output_file: str,
    teencode_file: str,
    text_column: str = "text",
    filter_non_vn: bool = True
) -> Tuple[int, int, int, int]:
    """
    Preprocess CSV file:
    - Excludes non-Vietnamese posts if filter_non_vn is True.
    - Removes icons / emojis.
    - Converts teencode to standard words in the specified text column.
    """
    ensure_teencode_file(teencode_file)

    print(f"Reading input file:      {input_file}")
    print(f"Teencode dictionary:     {teencode_file}")
    print(f"Target text column:      '{text_column}'")
    print(f"Filter non-Vietnamese:   {filter_non_vn}")

    teencode_dict = load_teencode_dict(teencode_file)
    print(f"Loaded {len(teencode_dict)} teencode dictionary mapping(s).")

    teencode_pattern = build_teencode_pattern(teencode_dict)

    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found at: {input_file}")

    total_rows = 0
    excluded_non_vn_count = 0
    kept_rows = 0
    icons_removed_count = 0
    teencode_converted_count = 0

    os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
    temp_output_file = output_file + ".tmp" if os.path.abspath(input_file) == os.path.abspath(output_file) else output_file

    start_time = time.time()

    with open(input_file, "r", newline="", encoding="utf-8-sig", errors="replace") as in_f:
        reader = csv.DictReader(in_f)
        fieldnames = reader.fieldnames

        if not fieldnames or text_column not in fieldnames:
            raise ValueError(
                f"Column '{text_column}' not found in CSV. Available columns: {fieldnames}"
            )

        with open(temp_output_file, "w", newline="", encoding="utf-8-sig") as out_f:
            writer = csv.DictWriter(out_f, fieldnames=fieldnames)
            writer.writeheader()

            for row in reader:
                total_rows += 1
                orig_text = row.get(text_column) or ""

                # Step 1: Check language filter
                if filter_non_vn and not is_vietnamese(orig_text):
                    excluded_non_vn_count += 1
                    continue

                kept_rows += 1

                # Step 2: Remove icons
                no_icons = remove_icons(orig_text)
                if no_icons != orig_text:
                    icons_removed_count += 1

                # Step 3: Convert teencode
                processed_text = convert_teencode(no_icons, teencode_dict, teencode_pattern)
                if processed_text != no_icons:
                    teencode_converted_count += 1

                row[text_column] = processed_text
                writer.writerow(row)

    if temp_output_file != output_file:
        os.replace(temp_output_file, output_file)

    elapsed = time.time() - start_time
    print(f"\nProcessing Summary:")
    print(f"  - Total rows read:              {total_rows:,}")
    if filter_non_vn:
        print(f"  - Non-Vietnamese rows excluded: {excluded_non_vn_count:,}")
        print(f"  - Vietnamese rows kept:         {kept_rows:,}")
    print(f"  - Rows with icons removed:      {icons_removed_count:,}")
    print(f"  - Rows with teencode converted: {teencode_converted_count:,}")
    print(f"  - Time elapsed:                 {elapsed:.2f} seconds")
    print(f"  - Output saved to:              {os.path.abspath(output_file)}")

    return total_rows, excluded_non_vn_count, icons_removed_count, teencode_converted_count

In [ ]:
# Run preprocessing on combined.csv -> combined_preprocessed.csv
import pandas as pd

preprocess_csv(
    input_file=RAW_COMBINED_CSV,
    output_file=PREPROCESSED_CSV,
    teencode_file=TEENCODE_FILE,
    text_column=TEXT_COLUMN,
    filter_non_vn=FILTER_NON_VN,
)

df_preprocessed = pd.read_csv(PREPROCESSED_CSV)
print(f"\nPreprocessed dataset shape: {df_preprocessed.shape}")
df_preprocessed.head(5)

## 4. Step 3 — Word Segmentation & Sentence Embeddings (`embedding/generate_embeddings.py`)

Tokenizes preprocessed Vietnamese text using `pyvi.ViTokenizer` and encodes dense 768D sentence embeddings using `dangvantuan/vietnamese-embedding`.

In [ ]:
import json
import numpy as np
import torch
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer


def get_default_device() -> str:
    """Select optimal device: CUDA, MPS (Apple Silicon), or CPU."""
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def tokenize_texts(texts: List[str]) -> List[str]:
    """Tokenize Vietnamese texts into segmented words using pyvi.ViTokenizer."""
    tokenized = []
    for text in texts:
        if not text or not str(text).strip():
            tokenized.append("")
        else:
            tokenized.append(ViTokenizer.tokenize(str(text)))
    return tokenized


def generate_embeddings(
    input_file: str,
    output_dir: str,
    model_name: str = "dangvantuan/vietnamese-embedding",
    text_column: str = "text",
    batch_size: int = 32,
    device: Optional[str] = None,
    normalize: bool = True
) -> Tuple[np.ndarray, List[str]]:
    """
    Load data, tokenize with PyVi, compute sentence embeddings, and save results.
    """
    if device is None:
        device = get_default_device()

    print("============================================================")
    print("Vietnamese Sentence Embedding Pipeline")
    print("============================================================")
    print(f"Input file:        {input_file}")
    print(f"Output directory:  {output_dir}")
    print(f"Model name:        {model_name}")
    print(f"Target column:     '{text_column}'")
    print(f"Device:            {device}")
    print(f"Batch size:        {batch_size}")
    print(f"Normalize vectors: {normalize}")
    print("------------------------------------------------------------")

    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")

    os.makedirs(output_dir, exist_ok=True)

    # 1. Read Input CSV
    start_time = time.time()
    rows = []
    with open(input_file, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        if text_column not in fieldnames:
            raise ValueError(f"Column '{text_column}' not found. Available: {fieldnames}")
        for r in reader:
            rows.append(r)

    total_rows = len(rows)
    print(f"Loaded {total_rows:,} rows from CSV.")
    raw_texts = [r.get(text_column, "") for r in rows]

    # 2. Tokenize with PyVi
    print("\nStep 1: Tokenizing with PyVi ViTokenizer...")
    tok_start = time.time()
    tokenized_texts = tokenize_texts(raw_texts)
    tok_time = time.time() - tok_start
    print(f"Tokenization completed in {tok_time:.2f} seconds ({total_rows / max(tok_time, 0.001):.1f} texts/sec).")

    # Display before & after samples
    if total_rows > 0:
        print("\nTokenization Samples:")
        for idx in sorted(set([0, min(1, total_rows - 1), min(2, total_rows - 1)])):
            sample_raw = raw_texts[idx].replace("\n", " ")[:70]
            sample_tok = tokenized_texts[idx].replace("\n", " ")[:70]
            print(f"  [{idx}] Raw:       {sample_raw}...")
            print(f"      Tokenized: {sample_tok}...")

    # 3. Load Embedding Model
    print(f"\nStep 2: Loading embedding model '{model_name}' on {device}...")
    load_start = time.time()
    model = SentenceTransformer(model_name, device=device)
    load_time = time.time() - load_start
    print(f"Model loaded in {load_time:.2f} seconds.")

    # 4. Compute Embeddings
    print(f"\nStep 3: Encoding {total_rows:,} sentences (batch size = {batch_size})...")
    encode_start = time.time()
    embeddings = model.encode(
        tokenized_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=normalize,
        convert_to_numpy=True
    )
    encode_time = time.time() - encode_start
    print(f"Encoding finished in {encode_time:.2f} seconds ({total_rows / max(encode_time, 0.001):.1f} texts/sec).")
    print(f"Embedding shape: {embeddings.shape} (dtype: {embeddings.dtype})")

    # 5. Save Results
    print(f"\nStep 4: Saving results into '{output_dir}'...")

    npy_path = os.path.join(output_dir, "embeddings.npy")
    np.save(npy_path, embeddings)
    print(f"  - Saved embeddings array:  {npy_path} ({os.path.getsize(npy_path) / (1024 * 1024):.2f} MB)")

    csv_path = os.path.join(output_dir, "tokenized_posts.csv")
    new_fieldnames = fieldnames.copy()
    if "tokenized_text" not in new_fieldnames:
        text_idx = new_fieldnames.index(text_column)
        new_fieldnames.insert(text_idx + 1, "tokenized_text")

    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=new_fieldnames)
        writer.writeheader()
        for r, tok in zip(rows, tokenized_texts):
            row_dict = r.copy()
            row_dict["tokenized_text"] = tok
            writer.writerow(row_dict)
    print(f"  - Saved tokenized CSV:     {csv_path} ({os.path.getsize(csv_path) / 1024:.1f} KB)")

    npz_path = os.path.join(output_dir, "embeddings_with_metadata.npz")
    np.savez_compressed(
        npz_path,
        embeddings=embeddings,
        tokenized_texts=np.array(tokenized_texts, dtype=object),
        urls=np.array([r.get("url", "") for r in rows], dtype=object)
    )
    print(f"  - Saved compressed NPZ:    {npz_path} ({os.path.getsize(npz_path) / (1024 * 1024):.2f} MB)")

    info_path = os.path.join(output_dir, "embedding_info.json")
    info = {
        "model_name": model_name,
        "embedding_dimension": int(embeddings.shape[1]),
        "total_sentences": int(embeddings.shape[0]),
        "dtype": str(embeddings.dtype),
        "normalized": normalize,
        "device": device,
        "input_file": os.path.abspath(input_file),
        "tokenization_time_seconds": round(tok_time, 3),
        "encoding_time_seconds": round(encode_time, 3),
        "total_elapsed_seconds": round(time.time() - start_time, 3),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")
    }
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)
    print(f"  - Saved embedding metadata:{info_path}")

    total_elapsed = time.time() - start_time
    print(f"\nAll operations completed successfully in {total_elapsed:.2f} seconds!")
    return embeddings, tokenized_texts

In [ ]:
# Run Vietnamese tokenization and sentence embedding generation
embeddings, tokenized_texts = generate_embeddings(
    input_file=PREPROCESSED_CSV,
    output_dir=EMBEDDING_DIR,
    model_name=MODEL_NAME,
    text_column=TEXT_COLUMN,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    normalize=NORMALIZE_EMBEDDINGS,
)

EMBEDDINGS_PATH = os.path.join(EMBEDDING_DIR, "embeddings.npy")
TOKENIZED_CSV_PATH = os.path.join(EMBEDDING_DIR, "tokenized_posts.csv")

## 5. Step 4 — Evaluation Metrics & Topic Extraction (`clustering/evaluation/`)

Combines:
- `clustering/evaluation/metrics.py` (`evaluate_clustering`)
- `clustering/evaluation/topic_extractor.py` (`VIETNAMESE_STOPWORDS`, `ClassTFIDF`, `extract_cluster_topics`)

In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def evaluate_clustering(
    embeddings: np.ndarray,
    labels: np.ndarray,
    distance_metric: str = "cosine"
) -> Dict[str, Optional[float]]:
    """
    Compute intrinsic clustering evaluation metrics.
    Handles noise/outlier labels (-1) properly by evaluating only on non-noise points
    when noise is present.
    """
    total_samples = len(labels)
    unique_labels = set(labels)
    non_noise_mask = labels != -1
    n_clusters = len(unique_labels - {-1})
    outlier_count = int(np.sum(~non_noise_mask))
    outlier_ratio = float(outlier_count / total_samples)

    eval_embeddings = embeddings[non_noise_mask]
    eval_labels = labels[non_noise_mask]

    # Cluster sizes and balance
    cluster_sizes = [int(np.sum(eval_labels == c)) for c in set(eval_labels)]
    min_size = min(cluster_sizes) if cluster_sizes else 0
    max_size = max(cluster_sizes) if cluster_sizes else 0
    balance_ratio = round(min_size / max(1, max_size), 4)

    results = {
        "n_clusters": n_clusters,
        "total_samples": total_samples,
        "outlier_count": outlier_count,
        "outlier_ratio": round(outlier_ratio, 4),
        "min_cluster_size": min_size,
        "max_cluster_size": max_size,
        "balance_ratio": balance_ratio,
        "silhouette": None,
        "davies_bouldin": None,
        "calinski_harabasz": None,
        "intra_cluster_cosine": None
    }

    if n_clusters < 2 or len(set(eval_labels)) < 2:
        return results

    try:
        sil = silhouette_score(eval_embeddings, eval_labels, metric=distance_metric)
        results["silhouette"] = round(float(sil), 4)
    except Exception:
        results["silhouette"] = None

    try:
        db = davies_bouldin_score(eval_embeddings, eval_labels)
        results["davies_bouldin"] = round(float(db), 4)
    except Exception:
        results["davies_bouldin"] = None

    try:
        ch = calinski_harabasz_score(eval_embeddings, eval_labels)
        results["calinski_harabasz"] = round(float(ch), 2)
    except Exception:
        results["calinski_harabasz"] = None

    try:
        sims = []
        for c in set(eval_labels):
            cluster_pts = eval_embeddings[eval_labels == c]
            if len(cluster_pts) > 1:
                centroid = np.mean(cluster_pts, axis=0)
                norm = np.linalg.norm(centroid)
                if norm > 0:
                    centroid /= norm
                    cluster_sims = np.dot(cluster_pts, centroid)
                    sims.append(np.mean(cluster_sims))
        if sims:
            results["intra_cluster_cosine"] = round(float(np.mean(sims)), 4)
    except Exception:
        results["intra_cluster_cosine"] = None

    return results

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Comprehensive Vietnamese Stopwords Categorized for Topic Extraction
VIETNAMESE_STOPWORDS = {
    # 1. Standard grammatical particles, prepositions, and conjunctions
    'và', 'mà', 'là', 'thì', 'có', 'đó', 'đi', 'về', 'ra', 'vào',
    'ở', 'trên', 'dưới', 'với', 'cho', 'để', 'từ', 'đến', 'khi', 'nếu', 'vì', 'như',
    'tôi', 'ta', 'mình', 'bạn', 'em', 'anh', 'chị', 'ông', 'bà', 'cô', 'chú', 'bác',
    'nó', 'họ', 'chúng', 'ai', 'gì', 'sao', 'đâu', 'nào', 'mấy', 'bao', 'nhiêu', 'này',
    'kia', 'đây', 'đấy', 'rồi', 'chưa', 'đang', 'sẽ', 'đã', 'cũng', 'vẫn', 'cứ', 'lại',
    'chỉ', 'mới', 'rất', 'quá', 'lắm', 'hơn', 'nhất', 'được', 'bị', 'phải', 'nên', 'cần',
    'muốn', 'biết', 'thấy', 'nghe', 'nói', 'bảo', 'làm', 'việc', 'chuyện', 'cái', 'con',
    'thôi', 'luôn', 'ngay', 'cả', 'tất', 'mọi', 'mỗi', 'từng', 'vài', 'một', 'hai', 'ba',
    'không', 'chẳng', 'ơi', 'ạ', 'nhé', 'nha', 'chứ', 'người', 'của', 'các', 'nhưng',
    'trong', 'những', 'theo', 'do', 'còn', 'vừa', 'hay', 'nữa', 'vậy', 'khác',
    'bởi', 'bởi_vì', 'tuy', 'dù', 'mặc_dù', 'ngoài_ra', 'thậm_chí', 'đều', 'chính',

    # 2. Web, scraping, and social media platform artifacts
    'translate', 'com', 'vn', 'net', 'org', 'http', 'https', 'www', 'html', 'link', 'bio',
    'thread', 'threads', 'post', 'cmt', 'comment', 'up', 'keonho', 'inbox', 'ib', 'app',
    'page', 'acc', 'fl', 'follow', 'admin', 'ad', 'sub', 'share', 'tag', 'clip', 'video_link',

    # 3. Conversational fillers, common verbs, and epistemic markers
    'dùng', 'xong', 'nhìn', 'lên', 'ngồi', 'phát', 'toàn', 'nhiều', 'ít', 'thích',
    'gặp', 'hỏi', 'tặng', 'mua', 'xem', 'chơi', 'ăn', 'uống', 'có_thể', 'làm_sao',
    'thực_sự', 'hãy', 'bắt_đầu', 'cảm_thấy', 'nghĩ', 'tự', 'make', 'you', 'help',
    'just', 'nội', 'thứ', 'bên', 'tìm', 'tới', 'nay', 'nhau', 'xíu', 'biết_đâu',
    'vụ', 'thường_tình', 'tròn', 'để_ý', 'bảo_sao', 'ra_sao', 'tính', 'cho_hỏi',
    'ai_biết', 'chắc', 'hình_như', 'dường_như', 'coi', 'đem', 'mang', 'giữ',
    'hết', 'chung', 'riêng', 'cùng', 'nào_đó', 'ai_đó',

    # 4. Temporal words that do not signify topical themes
    'ngày', 'hôm_nay', 'hôm_qua', 'hôm', 'sáng', 'trưa', 'chiều', 'tối', 'đêm',
    'năm', 'tháng', 'tuần', 'giờ', 'lúc', 'lần', 'lần_đầu', 'dạo_này', 'bây_giờ',
    'ngày_xưa', 'thời_gian', 'mấy_hôm', 'nay', 'bữa', 'bữa_nay',

    # 5. Colloquial interjections, profanities, and chat particles
    'á', 'à', 'ơi', 'ạ', 'nè', 'nhen', 'nhở', 'nha', 'nhé', 'hả', 'ủa', 'thế',
    'ha', 'hì', 'hic', 'ôi', 'trời', 'má', 'vãi', 'lồn', 'vl', 'vcl', 'vlll', 'con_mẹ',
    'mày', 'tao', 'bọn', 'đứa', 'ấy', 'mừi', 'tủi', 'nhỉ', 'ghê', 'nhaaa', 'nhaaaa',
    'địt', 'đụ', 'đm', 'đmm', 'vcll', 'đéo', 'hộ', 'sời', 'kìa', 'đấy_nhé', 'ổng',
    'bả', 'hí', 'haha', 'hihi', 'huhu', 'he_he', 'kaka', 'chèn_ơi', 'trời_ơi',
    'chèn', 'úi', 'ối',

    # 6. Quantifiers, counters, and generic reference words
    'cuốn', 'cốc', 'chiếc', 'bức', 'đoạn', 'phần', 'kiểu', 'khách_hàng', 'anh_em',
    'mọi_người', 'cả_nhà', 'bản_thân', 'ai_đó', 'tô', 'miếng', 'bộ', 'loại',
    'món', 'chút', 'tí', 'chút_xíu'
}


class ClassTFIDF:
    """
    Class-based TF-IDF implementation (c-TF-IDF).
    Treats all texts in a single cluster as a unified document to find
    terms that uniquely distinguish that cluster across all clusters.
    """

    def __init__(
        self,
        stop_words=None,
        max_features: int = 5000,
        max_df: float = 0.70,
        min_df: int = 1
    ):
        self.stop_words = list(stop_words) if stop_words is not None else list(VIETNAMESE_STOPWORDS)
        self.max_features = max_features
        self.max_df = max_df
        self.min_df = min_df
        self.token_pattern = r"(?u)\b[a-zA-Z_À-ỹ]{2,}\b"
        self.vectorizer = CountVectorizer(
            stop_words=self.stop_words,
            max_features=self.max_features,
            max_df=self.max_df,
            min_df=self.min_df,
            token_pattern=self.token_pattern
        )
        self.words = []
        self.c_tf_idf = None

    def fit_transform(self, cluster_docs: List[str]) -> np.ndarray:
        """
        Compute c-TF-IDF matrix for cluster documents.
        W_{x, c} = ||tf_{x, c}|| * log(1 + A / f_x)
        where A is average words per cluster, f_x is frequency of word x across all classes.
        """
        raw_count_matrix = self.vectorizer.fit_transform(cluster_docs).toarray()
        raw_words = np.array(self.vectorizer.get_feature_names_out())

        valid_mask = np.array([
            not bool(re.search(r'\d', w)) and len(w) >= 2
            for w in raw_words
        ])

        if np.any(valid_mask):
            count_matrix = raw_count_matrix[:, valid_mask]
            self.words = raw_words[valid_mask]
        else:
            count_matrix = raw_count_matrix
            self.words = raw_words

        total_words_per_cluster = count_matrix.sum(axis=1, keepdims=True)
        total_words_per_cluster[total_words_per_cluster == 0] = 1

        tf = count_matrix / total_words_per_cluster
        f_x = count_matrix.sum(axis=0)
        avg_words = count_matrix.sum() / max(1, count_matrix.shape[0])
        idf = np.log(1 + (avg_words / np.maximum(1, f_x)))

        self.c_tf_idf = tf * idf
        return self.c_tf_idf

    def get_top_words(
        self,
        cluster_idx: int,
        top_n: int = 5,
        format_spaces: bool = True
    ) -> List[Tuple[str, float]]:
        """
        Get top N words and their c-TF-IDF scores for a specific cluster.
        If format_spaces=True, converts underscores in compound words back to spaces.
        """
        if self.c_tf_idf is None or cluster_idx >= self.c_tf_idf.shape[0]:
            return []
        scores = self.c_tf_idf[cluster_idx]
        top_indices = scores.argsort()[::-1][:top_n]
        results = []
        for i in top_indices:
            if scores[i] > 0:
                word = self.words[i]
                if format_spaces:
                    word = word.replace('_', ' ')
                results.append((word, float(scores[i])))
        return results


def extract_cluster_topics(
    texts: List[str],
    labels: np.ndarray,
    top_n_words: int = 4,
    format_spaces: bool = True
) -> Dict[int, Dict]:
    """
    Extract distinctive keywords and representative post for each cluster.
    """
    unique_labels = sorted(list(set(labels)))
    valid_labels = [c for c in unique_labels if c != -1]

    if not valid_labels:
        return {}

    cluster_docs = []
    for c in valid_labels:
        c_texts = [str(texts[i]) for i in range(len(texts)) if labels[i] == c]
        cluster_docs.append(" ".join(c_texts))

    c_tfidf = ClassTFIDF()
    try:
        c_tfidf.fit_transform(cluster_docs)
    except Exception:
        return {
            c: {
                "keywords": [f"topic_{c}"],
                "keyword_string": f"Topic {c}",
                "size": int(np.sum(labels == c))
            }
            for c in unique_labels
        }

    topics = {}
    for idx, c in enumerate(valid_labels):
        top_scored = c_tfidf.get_top_words(idx, top_n=top_n_words, format_spaces=format_spaces)
        keywords = [w for w, _ in top_scored]
        topics[c] = {
            "keywords": keywords,
            "keyword_string": ", ".join(keywords) if keywords else f"Cluster {c}",
            "size": int(np.sum(labels == c))
        }

    if -1 in unique_labels:
        topics[-1] = {
            "keywords": ["noise", "outlier"],
            "keyword_string": "Noise / Outliers",
            "size": int(np.sum(labels == -1))
        }

    return topics

## 6. Step 5 — Clustering Benchmark Suite (`clustering/run_benchmark.py`)

Benchmarks 5 distinct clustering algorithm families on the 768D Vietnamese embeddings:
1. **Spherical K-Means** (`k = 3..10`)
2. **Agglomerative Hierarchical Clustering** (`cosine/average` and `euclidean/ward`, `k = 4..8`)
3. **HDBSCAN** (Direct Euclidean on normalized vectors & UMAP-5D reduced)
4. **Gaussian Mixture Models (GMM)** with PCA-20D reduction (`k = 4..8`)
5. **BERTopic** (End-to-end topic modeling with UMAP + HDBSCAN)

Ranks all configurations using a composite score, extracts `c-TF-IDF` topic labels for the winning model, saves CSV/JSON/PNG/HTML artifacts, and displays the plots inline.

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import hdbscan
import umap
from bertopic import BERTopic


def format_hover_text(text: str, max_chars: int = 140) -> str:
    """Format snippet for hover text with line breaks."""
    if not text or (isinstance(text, float) and np.isnan(text)):
        return ""
    clean = str(text).replace("\n", " ").strip()
    if len(clean) > max_chars:
        clean = clean[:max_chars] + "..."
    words = clean.split()
    lines, curr_line = [], []
    curr_len = 0
    for w in words:
        if curr_len + len(w) > 48:
            lines.append(" ".join(curr_line))
            curr_line = [w]
            curr_len = len(w)
        else:
            curr_line.append(w)
            curr_len += len(w) + 1
    if curr_line:
        lines.append(" ".join(curr_line))
    return "<br>".join(lines)


def run_benchmark(
    embeddings_path: str,
    data_path: str,
    output_dir: str,
    random_state: int = 42,
    show_plots: bool = True
) -> Tuple[pd.DataFrame, object, Dict]:
    """
    Run full benchmarking suite across all clustering algorithm families.
    """
    os.makedirs(output_dir, exist_ok=True)
    start_total_time = time.time()

    print("==================================================================")
    print("CLUSTERING BENCHMARK: 768D VIETNAMESE SENTENCE EMBEDDINGS")
    print("==================================================================")
    print(f"Embeddings:  {embeddings_path}")
    print(f"Data:        {data_path}")
    print(f"Output Dir:  {output_dir}")
    print("------------------------------------------------------------------")

    # 1. Load Data
    embeddings = np.load(embeddings_path)
    df = pd.read_csv(data_path)
    total_samples = len(embeddings)
    print(f"Loaded {total_samples:,} embeddings with dimension {embeddings.shape[1]}.")
    print(f"Loaded {len(df):,} posts from CSV.")

    text_col = "tokenized_text" if "tokenized_text" in df.columns else "text"
    raw_texts = df["text"].fillna("").tolist()
    processed_texts = df[text_col].fillna("").tolist()

    all_results = []
    labels_dict = {}

    # ------------------------------------------------------------------
    # 1. SPHERICAL K-MEANS SWEEP
    # ------------------------------------------------------------------
    print("\n[1/5] Benchmarking Spherical K-Means...")
    k_range = [3, 4, 5, 6, 7, 8, 9, 10]
    for k in k_range:
        t0 = time.time()
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
        labels = kmeans.fit_predict(embeddings)
        runtime = time.time() - t0

        metrics = evaluate_clustering(embeddings, labels, distance_metric="cosine")
        config_name = f"KMeans (k={k})"
        labels_dict[config_name] = labels

        all_results.append({
            "algorithm_family": "Centroid-based",
            "algorithm": "K-Means",
            "config": f"k={k}",
            "name": config_name,
            "runtime_sec": round(runtime, 3),
            **metrics
        })
        print(f"  - {config_name:18}: Sil = {metrics['silhouette']:.4f}, DB = {metrics['davies_bouldin']:.4f}, CH = {metrics['calinski_harabasz']:.1f}")

    # ------------------------------------------------------------------
    # 2. AGGLOMERATIVE HIERARCHICAL CLUSTERING SWEEP
    # ------------------------------------------------------------------
    print("\n[2/5] Benchmarking Agglomerative Hierarchical Clustering...")
    for linkage, metric in [("average", "cosine"), ("ward", "euclidean")]:
        for k in [4, 5, 6, 7, 8]:
            t0 = time.time()
            if linkage == "ward":
                agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
            else:
                agg = AgglomerativeClustering(n_clusters=k, metric=metric, linkage=linkage)
            labels = agg.fit_predict(embeddings)
            runtime = time.time() - t0

            metrics = evaluate_clustering(embeddings, labels, distance_metric="cosine")
            config_name = f"Agglomerative ({linkage}, k={k})"
            labels_dict[config_name] = labels

            all_results.append({
                "algorithm_family": "Hierarchical",
                "algorithm": f"Agglomerative-{linkage}",
                "config": f"linkage={linkage}, k={k}",
                "name": config_name,
                "runtime_sec": round(runtime, 3),
                **metrics
            })
            print(f"  - {config_name:30}: Sil = {metrics['silhouette']:.4f}, DB = {metrics['davies_bouldin']:.4f}")

    # ------------------------------------------------------------------
    # 3. HDBSCAN (DIRECT & UMAP)
    # ------------------------------------------------------------------
    print("\n[3/5] Benchmarking HDBSCAN...")
    for min_size in [10, 15, 20]:
        t0 = time.time()
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            metric="euclidean",
            cluster_selection_method="eom"
        )
        labels = clusterer.fit_predict(embeddings)
        runtime = time.time() - t0

        metrics = evaluate_clustering(embeddings, labels, distance_metric="cosine")
        config_name = f"HDBSCAN-Direct (min_size={min_size})"
        labels_dict[config_name] = labels

        all_results.append({
            "algorithm_family": "Density-based",
            "algorithm": "HDBSCAN-Direct",
            "config": f"min_cluster_size={min_size}",
            "name": config_name,
            "runtime_sec": round(runtime, 3),
            **metrics
        })
        sil_str = f"{metrics['silhouette']:.4f}" if metrics['silhouette'] is not None else "N/A"
        print(f"  - {config_name:32}: Clusters = {metrics['n_clusters']}, Outliers = {metrics['outlier_count']}, Sil = {sil_str}")

    print("  Fitting UMAP (768D -> 5D)...")
    t_umap = time.time()
    umap_model = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=random_state)
    umap_embs_5d = umap_model.fit_transform(embeddings)
    print(f"  UMAP 5D completed in {time.time() - t_umap:.2f}s.")

    for min_size in [10, 15, 20, 25]:
        t0 = time.time()
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            metric="euclidean",
            cluster_selection_method="eom"
        )
        labels = clusterer.fit_predict(umap_embs_5d)
        runtime = time.time() - t0

        metrics = evaluate_clustering(embeddings, labels, distance_metric="cosine")
        config_name = f"UMAP+HDBSCAN (min_size={min_size})"
        labels_dict[config_name] = labels

        all_results.append({
            "algorithm_family": "Density-based",
            "algorithm": "UMAP+HDBSCAN",
            "config": f"n_dim=5, min_cluster_size={min_size}",
            "name": config_name,
            "runtime_sec": round(runtime, 3),
            **metrics
        })
        sil_str = f"{metrics['silhouette']:.4f}" if metrics['silhouette'] is not None else "N/A"
        print(f"  - {config_name:32}: Clusters = {metrics['n_clusters']}, Outliers = {metrics['outlier_count']}, Sil = {sil_str}")

    # ------------------------------------------------------------------
    # 4. GAUSSIAN MIXTURE MODELS (GMM via PCA 20D)
    # ------------------------------------------------------------------
    print("\n[4/5] Benchmarking Gaussian Mixture Models (GMM)...")
    pca_20 = PCA(n_components=20, random_state=random_state)
    embs_pca20 = pca_20.fit_transform(embeddings)

    for k in [4, 5, 6, 7, 8]:
        t0 = time.time()
        gmm = GaussianMixture(n_components=k, covariance_type="diag", random_state=random_state)
        labels = gmm.fit_predict(embs_pca20)
        runtime = time.time() - t0

        metrics = evaluate_clustering(embeddings, labels, distance_metric="cosine")
        config_name = f"GMM-PCA20 (k={k})"
        labels_dict[config_name] = labels

        all_results.append({
            "algorithm_family": "Probabilistic",
            "algorithm": "GMM",
            "config": f"k={k}, cov=diag",
            "name": config_name,
            "runtime_sec": round(runtime, 3),
            **metrics
        })
        print(f"  - {config_name:20}: Sil = {metrics['silhouette']:.4f}, DB = {metrics['davies_bouldin']:.4f}")

    # ------------------------------------------------------------------
    # 5. BERTopic (END-TO-END TOPIC MODELING)
    # ------------------------------------------------------------------
    print("\n[5/5] Benchmarking BERTopic...")
    for min_size in [10, 15, 20]:
        t0 = time.time()
        umap_bt = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=random_state)
        hdbscan_bt = hdbscan.HDBSCAN(min_cluster_size=min_size, metric="euclidean", cluster_selection_method="eom")
        topic_model = BERTopic(
            umap_model=umap_bt,
            hdbscan_model=hdbscan_bt,
            calculate_probabilities=False,
            verbose=False
        )
        topics, _ = topic_model.fit_transform(raw_texts, embeddings=embeddings)
        runtime = time.time() - t0

        labels = np.array(topics)
        metrics = evaluate_clustering(embeddings, labels, distance_metric="cosine")
        config_name = f"BERTopic (min_size={min_size})"
        labels_dict[config_name] = labels

        all_results.append({
            "algorithm_family": "Topic Modeling",
            "algorithm": "BERTopic",
            "config": f"min_topic_size={min_size}",
            "name": config_name,
            "runtime_sec": round(runtime, 3),
            **metrics
        })
        sil_str = f"{metrics['silhouette']:.4f}" if metrics['silhouette'] is not None else "N/A"
        print(f"  - {config_name:24}: Topics = {metrics['n_clusters']}, Outliers = {metrics['outlier_count']}, Sil = {sil_str}")

    # ------------------------------------------------------------------
    # 6. EVALUATE & RANK ALL CONFIGURATIONS
    # ------------------------------------------------------------------
    results_df = pd.DataFrame(all_results)

    valid_mask = results_df["silhouette"].notnull() & (results_df["n_clusters"] >= 2)
    scores = []
    for idx, row in results_df.iterrows():
        if not valid_mask[idx]:
            scores.append(-999.0)
            continue
        sil = row["silhouette"] or 0
        db = row["davies_bouldin"] or 10.0
        outlier_pen = (row["outlier_ratio"] or 0) * 1.5
        balance = row.get("balance_ratio", 0) or 0
        max_ratio = (row.get("max_cluster_size", 0) or 0) / total_samples
        degenerate_pen = 1.0 if max_ratio > 0.70 else 0.0

        score = (sil * 2.5) - (db * 0.10) + (balance * 0.15) - outlier_pen - degenerate_pen
        scores.append(round(score, 4))

    results_df["composite_score"] = scores
    results_df = results_df.sort_values(by="composite_score", ascending=False).reset_index(drop=True)
    results_df["rank"] = range(1, len(results_df) + 1)

    csv_path = os.path.join(output_dir, "benchmark_summary.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"\nSaved benchmark table to: {csv_path}")

    best_row = results_df.iloc[0]
    best_name = best_row["name"]
    best_labels = labels_dict[best_name]
    print("\n==================================================================")
    print(f"WINNING ALGORITHM: {best_name}")
    print(f"  - Family:           {best_row['algorithm_family']}")
    print(f"  - Number of topics: {best_row['n_clusters']}")
    print(f"  - Silhouette Score: {best_row['silhouette']}")
    print(f"  - Davies-Bouldin:   {best_row['davies_bouldin']}")
    print(f"  - Outliers:         {best_row['outlier_count']} ({best_row['outlier_ratio']*100:.1f}%)")
    print(f"  - Composite Score:  {best_row['composite_score']}")
    print("==================================================================")

    # ------------------------------------------------------------------
    # 7. EXTRACT BEST MODEL TOPICS & ATTACH TO DATASET
    # ------------------------------------------------------------------
    print(f"\nExtracting c-TF-IDF topic keywords for {best_name}...")
    topics_info = extract_cluster_topics(processed_texts, best_labels, top_n_words=5, format_spaces=True)

    cluster_labels_text = []
    for lbl in best_labels:
        if lbl in topics_info:
            cluster_labels_text.append(f"Topic {lbl}: {topics_info[lbl]['keyword_string']}")
        else:
            cluster_labels_text.append(f"Topic {lbl}")

    df["cluster_id"] = best_labels
    df["topic_label"] = cluster_labels_text

    pca_2d = PCA(n_components=2, random_state=random_state)
    coords_2d = pca_2d.fit_transform(embeddings)
    df["dim_1"] = coords_2d[:, 0]
    df["dim_2"] = coords_2d[:, 1]

    clustered_csv_path = os.path.join(output_dir, "clustered_posts.csv")
    df.to_csv(clustered_csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved clustered dataset to: {clustered_csv_path}")

    # ------------------------------------------------------------------
    # 8. GENERATE COMPARATIVE VISUALIZATIONS
    # ------------------------------------------------------------------
    print("\nGenerating visualization artifacts...")

    # 8a. Metric comparison chart across top models
    metric_plot_path = os.path.join(output_dir, "metric_comparison.png")
    top_models = results_df.head(8)

    fig, ax1 = plt.subplots(figsize=(12, 6), dpi=150)
    fig.patch.set_facecolor("#FFFFFF")
    ax1.set_facecolor("#F8FAFC")

    x = np.arange(len(top_models))
    width = 0.35

    sil_vals = top_models["silhouette"].fillna(0).tolist()
    db_vals = top_models["davies_bouldin"].fillna(0).tolist()

    ax1.bar(x - width/2, sil_vals, width, label="Silhouette Score (↑ higher is better)", color="#2563EB", alpha=0.85)
    ax1.set_ylabel("Silhouette Score", color="#2563EB", fontsize=11, fontweight="bold")
    ax1.tick_params(axis="y", labelcolor="#2563EB")
    ax1.set_xticks(x)
    ax1.set_xticklabels(top_models["name"], rotation=30, ha="right", fontsize=9, fontweight="medium")

    ax2 = ax1.twinx()
    ax2.bar(x + width/2, db_vals, width, label="Davies-Bouldin Index (↓ lower is better)", color="#DB2777", alpha=0.85)
    ax2.set_ylabel("Davies-Bouldin Index", color="#DB2777", fontsize=11, fontweight="bold")
    ax2.tick_params(axis="y", labelcolor="#DB2777")

    plt.title("Clustering Quality Comparison Across Top Candidate Algorithms", fontsize=13, fontweight="bold", pad=15)
    ax1.grid(True, linestyle="--", alpha=0.3, color="#CBD5E1")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", framealpha=0.9)

    plt.tight_layout()
    plt.savefig(metric_plot_path, dpi=300)
    if show_plots:
        plt.show()
    else:
        plt.close()
    print(f"  - Saved metric comparison plot: {metric_plot_path}")

    # 8b. Best model 2D scatter plot
    best_plot_path = os.path.join(output_dir, "best_model_clusters.png")
    fig, ax = plt.subplots(figsize=(12, 8), dpi=150)
    fig.patch.set_facecolor("#FFFFFF")
    ax.set_facecolor("#F8FAFC")

    unique_best = sorted(list(set(best_labels)))
    colors = ["#2563EB", "#7C3AED", "#DB2777", "#059669", "#D97706", "#DC2626", "#0891B2", "#4F46E5", "#0D9488", "#EA580C"]

    for i, c in enumerate(unique_best):
        mask = best_labels == c
        color = "#94A3B8" if c == -1 else colors[i % len(colors)]
        label_text = f"Noise / Outliers ({np.sum(mask)})" if c == -1 else f"T{c}: {topics_info.get(c, {}).get('keyword_string', '')} ({np.sum(mask)})"
        ax.scatter(
            coords_2d[mask, 0],
            coords_2d[mask, 1],
            c=color,
            label=label_text,
            alpha=0.4 if c == -1 else 0.75,
            edgecolors="white" if c != -1 else "none",
            linewidth=0.5,
            s=25 if c == -1 else 45
        )

        if c != -1:
            cx, cy = np.mean(coords_2d[mask, 0]), np.mean(coords_2d[mask, 1])
            ax.text(
                cx, cy, f"T{c}",
                fontsize=9, fontweight="bold", color="#FFFFFF",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.3", facecolor=color, edgecolor="#0F172A", alpha=0.9)
            )

    ax.set_title(
        f"Best Model Clustering: {best_name}\n({best_row['n_clusters']} Semantic Topics, N={total_samples:,})",
        fontsize=14, fontweight="bold", pad=15
    )
    ax.set_xlabel(f"Dimension 1 (PCA {pca_2d.explained_variance_ratio_[0]*100:.1f}%)", fontsize=11, color="#334155")
    ax.set_ylabel(f"Dimension 2 (PCA {pca_2d.explained_variance_ratio_[1]*100:.1f}%)", fontsize=11, color="#334155")
    ax.grid(True, linestyle="--", alpha=0.4, color="#CBD5E1")
    ax.legend(title="Discovered Topics", loc="upper right", framealpha=0.92, fontsize=8.5)

    plt.tight_layout()
    plt.savefig(best_plot_path, dpi=300)
    if show_plots:
        plt.show()
    else:
        plt.close()
    print(f"  - Saved best model plot:        {best_plot_path}")

    # 8c. Interactive Plotly Dashboard
    interactive_path = os.path.join(output_dir, "best_model_interactive.html")
    df["hover_snippet"] = [format_hover_text(t) for t in df["text"].fillna("")]

    fig_interactive = px.scatter(
        df,
        x="dim_1",
        y="dim_2",
        color="topic_label",
        hover_data={
            "dim_1": False,
            "dim_2": False,
            "topic_label": True,
            "hover_snippet": True,
            "likes": True if "likes" in df.columns else False,
            "comments": True if "comments" in df.columns else False
        },
        title=f"Interactive Cluster Explorer - Best Model: {best_name}",
        template="plotly_white"
    )

    fig_interactive.update_traces(
        marker=dict(size=7, opacity=0.8, line=dict(width=0.5, color="white")),
        hovertemplate="<b>%{customdata[0]}</b><br><br>%{customdata[1]}<extra></extra>"
    )

    fig_interactive.update_layout(
        legend=dict(
            title=dict(text="<b>Topics</b>"),
            orientation="h",
            yanchor="bottom",
            y=-0.35,
            xanchor="center",
            x=0.5
        ),
        margin=dict(l=40, r=40, t=60, b=120),
        height=750
    )

    fig_interactive.write_html(interactive_path, include_plotlyjs="cdn")
    print(f"  - Saved interactive dashboard:   {interactive_path}")

    # Save summary JSON
    summary_json_path = os.path.join(output_dir, "benchmark_summary.json")
    summary_data = {
        "winning_model": best_name,
        "winning_family": str(best_row["algorithm_family"]),
        "n_clusters": int(best_row["n_clusters"]),
        "silhouette": float(best_row["silhouette"]) if pd.notnull(best_row["silhouette"]) else None,
        "davies_bouldin": float(best_row["davies_bouldin"]) if pd.notnull(best_row["davies_bouldin"]) else None,
        "calinski_harabasz": float(best_row["calinski_harabasz"]) if pd.notnull(best_row["calinski_harabasz"]) else None,
        "outlier_count": int(best_row["outlier_count"]),
        "topics": {str(k): v for k, v in topics_info.items()},
        "total_models_evaluated": len(results_df),
        "total_elapsed_seconds": round(time.time() - start_total_time, 2)
    }
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(summary_data, f, indent=2, ensure_ascii=False)
    print(f"  - Saved summary JSON:            {summary_json_path}")

    print(f"\nBenchmark completed successfully in {time.time() - start_total_time:.2f} seconds!")
    return results_df, fig_interactive, summary_data

In [ ]:
# Run the full clustering benchmark and display results
results_df, fig_interactive, summary_data = run_benchmark(
    embeddings_path=EMBEDDINGS_PATH,
    data_path=TOKENIZED_CSV_PATH,
    output_dir=CLUSTERING_OUTPUT_DIR,
    random_state=RANDOM_STATE,
    show_plots=True,
)

# Display ranked benchmark table
results_df[[
    "rank", "name", "algorithm_family", "n_clusters",
    "outlier_count", "silhouette", "davies_bouldin",
    "calinski_harabasz", "intra_cluster_cosine", "composite_score", "runtime_sec"
]]

In [ ]:
# Display interactive Plotly cluster explorer & discovered topic summary
print("Discovered Topics Summary:")
for topic_id, info in summary_data["topics"].items():
    print(f"  Topic {topic_id} ({info['size']} posts): {info['keyword_string']}")

fig_interactive.show()